#### This first part uses an API key to send a POST request to the Beiwe server and downloads participant data as a CSV file. It also imports the pandas library.

In [1]:
import requests
from io import BytesIO
study_id = "STUDY_ID" # enter study id here
# send a POST request to the Beiwe server to get participant table data
response = requests.post( 
    "https://studies.beiwe.org/get-participant-table-data/v1",
        data={
            "access_key": "ACCESS_KEY", # enter access key here
            "secret_key": "SECRET_KEY", # enter secret key here
            "data_format": "csv", # request data in csv data format
            "study_id": study_id,
    },
    allow_redirects=False,
)
# if the request succeeds, save the returned CSV to disk
if response.status_code == 200:
    participants_table = f"participant_table_{study_id}.csv"
    print(f"The data has been written to {participants_table}.")
    with open(participants_table, "wb") as f:
        f.write(response.content)
# otherwise, print the error returned by the server
else:
    print(response.content)
    print(f"There was an error downloading the data; The request returned an HTTP error of {response.status_code}")

b''
There was an error downloading the data; The request returned an HTTP error of 400


In [23]:
import pandas as pd
with open(participants_table, "rb") as f:
    df = pd.read_csv(f) # save the CSV data into a pandas DataFrame
    # this will not work unless the participants_table variable in cell above has been populated and the file exists


#### First, we will get the number of participants who are registered. (that is, they have a First Registration Date that is not "None")

In [11]:
registered_count = df["First Registration Date"].notna().sum() # count participants with a registration date
print(f"Participants registered: {registered_count}")

Participants registered: 31


#### Next, we will get the number of participants who are CURRENTLY registered. (anything BUT "Permanantly Retired" or "Not Registered")

In [12]:
currentlyreg_count = ((df["Status"] != "Permanently Retired") & (df["Status"] != "Not Registered")).sum() # count participants currently registered
print(f"Participants CURRENTLY registered: {currentlyreg_count}")

Participants CURRENTLY registered: 29


#### Now we will get counts of mobile device type (iOS and Android).

In [13]:
android_count = (df["OS Type"] == "ANDROID").sum() # count participants using Android devices
ios_count = (df["OS Type"] == "IOS").sum() # count participants using iOS devices
print(f"iOS count: {ios_count} --- Android count: {android_count}")

iOS count: 20 --- Android count: 11


#### Finally we will get a breakdown of status for each type besides "Permanently Retired" or "Not Registered", ie. how many are "Active (Just Now)", etc.

In [24]:
status_counts = (
    df.loc[
        ~df["Status"].isin(["Permanently Retired", "Not Registered"]), # exclude retired and not registered participants
        "Status"
    ]
    .value_counts()
)

print(
    " --- ".join(
        f"{status}: {count}"
        for status, count in status_counts.items() # format status counts for printing
    )
)


Inactive: 21 --- Active (just now): 6 --- Active (past week): 2


#### The following code will autosave these metrics as a PDF. 

In [27]:
from datetime import datetime

# this computes the metrics again to build the outputs for the PDF 
registered_count = df["First Registration Date"].notna().sum()

currentlyreg_count = (
    (~df["Status"].isin(["Permanently Retired", "Not Registered"]))
).sum()

android_count = (df["OS Type"] == "ANDROID").sum()
ios_count = (df["OS Type"] == "IOS").sum()

status_counts = (
    df.loc[~df["Status"].isin(["Permanently Retired", "Not Registered"]), "Status"]
    .value_counts()
)

status_line = " --- ".join(f"{status}: {count}" for status, count in status_counts.items())

# build output strings for the pdf
outputs = [
    f"Study ID: {study_id}",
    f"PDF created: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "",
    f"Participants registered: {registered_count}",
    f"Participants CURRENTLY registered: {currentlyreg_count}",
    f"iOS count: {ios_count} --- Android count: {android_count}",
    status_line,
]


# autosave pdf
pdf_file = autosave_outputs_to_pdf(study_id, outputs)
print(f"Autosaved PDF to: {pdf_file}")

Autosaved PDF to: outputs/study_XXb2Gm78TVBGesqyd6oPjjXQ_2026-01-24.pdf
